# Plant Disease Detection using YOLOv8 (Google Colab)

Complete workflow for setup, dataset checks, training, evaluation, prediction, and export.

In [ ]:
# 1) Check GPU (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

In [ ]:
# 2) Install dependencies
!pip install -q ultralytics pyyaml pandas opencv-python pillow matplotlib seaborn

import os
import random
import shutil
from pathlib import Path
import yaml
import pandas as pd
from ultralytics import YOLO

print('Ultralytics installed successfully')

## 3) Mount Google Drive (optional but recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to your dataset root in Drive
DATASET_ROOT = Path('/content/drive/MyDrive/PlantDoc-Dataset/dataset')
PROJECT_ROOT = Path('/content/PlantDoc-YOLOv8')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset root:', DATASET_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
# Optional: Upload dataset ZIP directly to Colab for training
# Expected ZIP structure example:
# dataset/
#   images/train ... + labels/train ...
# or flat format: dataset/images + dataset/labels

from google.colab import files
from zipfile import ZipFile

USE_UPLOAD = False  # Set True if you want to upload dataset zip from local PC
ZIP_NAME = 'dataset.zip'  # This should match your uploaded file name

if USE_UPLOAD:
    print('Upload your dataset ZIP now...')
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError('No file uploaded. Please upload a dataset zip file.')

    # Use first uploaded filename if ZIP_NAME not found
    zip_file = ZIP_NAME if ZIP_NAME in uploaded else list(uploaded.keys())[0]
    zip_path = Path('/content') / zip_file

    extract_root = Path('/content/uploaded_data')
    extract_root.mkdir(parents=True, exist_ok=True)

    with ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_root)

    # Auto-detect dataset root after extraction
    candidate_1 = extract_root / 'dataset'
    candidate_2 = extract_root

    if candidate_1.exists():
        DATASET_ROOT = candidate_1
    else:
        # Try finding folder that contains images and labels
        found = None
        for p in extract_root.rglob('*'):
            if p.is_dir() and (p / 'images').exists() and (p / 'labels').exists():
                found = p
                break
        DATASET_ROOT = found if found is not None else candidate_2

    print('Using uploaded dataset at:', DATASET_ROOT)
else:
    print('Using Drive dataset path:', DATASET_ROOT)

In [ ]:
# 4) Dataset structure check + optional auto-fix from flat format
# Supported input formats:
# A) Already split: dataset/images/train + dataset/labels/train (+ val/test)
# B) Flat: dataset/images + dataset/labels (this cell can split into train/val)

def count_files(folder, suffixes=None):
    if not folder.exists():
        return 0
    files = [p for p in folder.iterdir() if p.is_file()]
    if suffixes is None:
        return len(files)
    return sum(1 for p in files if p.suffix.lower() in suffixes)

img_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
images_dir = DATASET_ROOT / 'images'
labels_dir = DATASET_ROOT / 'labels'

has_split = (images_dir / 'train').exists() and (labels_dir / 'train').exists()
has_flat = any(p.is_file() and p.suffix.lower() in img_ext for p in images_dir.iterdir()) if images_dir.exists() else False

print('has_split =', has_split)
print('has_flat =', has_flat)

if has_split:
    for split in ['train', 'val', 'test']:
        ni = count_files(images_dir / split, img_ext)
        nl = count_files(labels_dir / split, {'.txt'})
        print(f'{split}: images={ni}, labels={nl}')
elif has_flat:
    print('Flat dataset detected. Splitting into train/val (80/20)...')
    all_images = sorted([p for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in img_ext])
    pairs = []
    for img in all_images:
        lbl = labels_dir / f'{img.stem}.txt'
        if lbl.exists():
            pairs.append((img, lbl))
    random.seed(42)
    random.shuffle(pairs)
    cut = int(0.8 * len(pairs))
    train_pairs, val_pairs = pairs[:cut], pairs[cut:]

    for split in ['train', 'val']:
        (DATASET_ROOT / split / 'images').mkdir(parents=True, exist_ok=True)
        (DATASET_ROOT / split / 'labels').mkdir(parents=True, exist_ok=True)

    for img, lbl in train_pairs:
        shutil.copy2(img, DATASET_ROOT / 'train' / 'images' / img.name)
        shutil.copy2(lbl, DATASET_ROOT / 'train' / 'labels' / lbl.name)
    for img, lbl in val_pairs:
        shutil.copy2(img, DATASET_ROOT / 'val' / 'images' / img.name)
        shutil.copy2(lbl, DATASET_ROOT / 'val' / 'labels' / lbl.name)

    print(f'Matched pairs: {len(pairs)}')
    print(f'Train: {len(train_pairs)}, Val: {len(val_pairs)}')
else:
    raise RuntimeError('Dataset format not recognized. Check dataset/images and dataset/labels.')

In [ ]:
# 5) Create/refresh data.yaml
# IMPORTANT: update class names to your exact dataset classes and order.

NAMES = [
    'Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf',
    'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot',
    'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight',
    'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf',
    'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf',
    'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot',
    'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus',
    'Tomato mold leaf', 'grape leaf', 'grape leaf black rot'
]

split_style = 'images_under_images' if (DATASET_ROOT / 'images' / 'train').exists() else 'direct_train_val'

if split_style == 'images_under_images':
    data_cfg = {
        'path': str(DATASET_ROOT),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test' if (DATASET_ROOT / 'images' / 'test').exists() else 'images/val',
        'nc': len(NAMES),
        'names': {i: n for i, n in enumerate(NAMES)}
    }
else:
    data_cfg = {
        'path': str(DATASET_ROOT),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'val/images',
        'nc': len(NAMES),
        'names': {i: n for i, n in enumerate(NAMES)}
    }

data_yaml_path = PROJECT_ROOT / 'data.yaml'
with open(data_yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

print('Saved:', data_yaml_path)
print(open(data_yaml_path, 'r', encoding='utf-8').read())

In [ ]:
# 6) Train YOLOv8 on T4
# You can switch model: yolov8n.pt / yolov8s.pt (s usually gives better accuracy)

MODEL_NAME = 'yolov8s.pt'
EPOCHS = 100
IMGSZ = 640
BATCH = 16

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(PROJECT_ROOT / 'runs'),
    name='detect/train',
    exist_ok=True,
    workers=2,
    verbose=True,
    plots=True
)
print('Training completed')
print('Run dir:', results.save_dir)

In [ ]:
# 7) Validate trained model
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
print('Best model path:', best_pt)

best_model = YOLO(str(best_pt))
metrics = best_model.val(data=str(data_yaml_path), split='val')
print('Validation done')
print(metrics)

In [ ]:
# 8) Single-image prediction (with disease summary logic)
CLASS_NAME_MAPPING = {
    'Potato leaf early blight': 'Early Blight',
    'Tomato Early blight leaf': 'Early Blight',
    'Potato leaf late blight': 'Late Blight',
    'Tomato leaf late blight': 'Late Blight',
    'Tomato Septoria leaf spot': 'Leaf Spot',
    'Bell_pepper leaf spot': 'Leaf Spot',
    'Tomato leaf bacterial spot': 'Leaf Spot',
}
PESTICIDE_RECOMMENDATIONS = {
    'Early Blight': 'Use Mancozeb or Chlorothalonil',
    'Late Blight': 'Use Copper fungicide',
    'Leaf Spot': 'Use Neem oil or fungicide spray',
}

def map_class_to_disease(raw_name: str) -> str:
    return CLASS_NAME_MAPPING.get(raw_name, raw_name)

def recommendation(disease: str) -> str:
    return PESTICIDE_RECOMMENDATIONS.get(disease, 'No specific recommendation available. Consult local agronomist.')

def infected_area_pct(x1, y1, x2, y2, w, h):
    box_area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    img_area = max(1.0, w * h)
    return max(0.0, min(100.0, (box_area / img_area) * 100.0))

def severity(conf_pct, area_pct):
    score = (conf_pct + area_pct) / 2.0
    if score < 10:
        return 'Low'
    if score < 30:
        return 'Medium'
    return 'Severe'

# Provide any test image path here
test_image = str((DATASET_ROOT / 'images' / 'val').glob('*').__iter__().__next__()) if (DATASET_ROOT / 'images' / 'val').exists() else str((DATASET_ROOT / 'val' / 'images').glob('*').__iter__().__next__())
pred = best_model.predict(source=test_image, conf=0.2, imgsz=960, augment=True, save=True, verbose=False)[0]

if pred.boxes is None or len(pred.boxes) == 0:
    print('No detection found')
else:
    names = best_model.names
    ih, iw = pred.orig_shape
    best_idx = int(pred.boxes.conf.argmax().item())
    b = pred.boxes[best_idx]
    cls_id = int(b.cls[0])
    conf = float(b.conf[0])
    conf_pct = conf * 100.0
    x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].tolist()]
    area_pct = infected_area_pct(x1, y1, x2, y2, iw, ih)
    health_pct = max(0.0, 100.0 - area_pct)
    disease = map_class_to_disease(str(names.get(cls_id, cls_id)))

    print(f'Disease: {disease}')
    print(f'Confidence: {conf_pct:.2f}%')
    print(f'Severity: {severity(conf_pct, area_pct)}')
    print(f'Health: {health_pct:.2f}%')
    print(f'Recommendation: {recommendation(disease)}')
    print(f'Bounding Box: ({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})')

In [ ]:
# 9) Export artifacts to Drive
EXPORT_DIR = Path('/content/drive/MyDrive/PlantDoc-Exports')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

run_dir = Path(results.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'

for f in [best_pt, last_pt, run_dir / 'results.csv', run_dir / 'results.png', run_dir / 'args.yaml']:
    if f.exists():
        shutil.copy2(f, EXPORT_DIR / f.name)
        print('Copied:', f.name)

print('Export complete ->', EXPORT_DIR)

## 10) Next Steps

1. Download `best.pt` from `PlantDoc-Exports`.
2. Place it in your local project at `runs/detect/train/weights/best.pt` (or use auto selection in your script).
3. Run local inference with your `model/predict.py`.